# Lesson 4 — Make it write

Generation is a loop: start from `<s>`, sample a next word from the model, append it, slide the context forward,
and repeat until `</s>`. It's the same autoregressive loop a GPT runs, just on counts here. Then I compare 1 word of
memory (bigram) with 2 (trigram).

In [1]:
import re, math, random
from collections import Counter, defaultdict

In [2]:
def to_sentences(text):
    return [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]

def to_tokens(sentence):
    return re.findall(r"[a-z']+", sentence.lower())

In [3]:
def build(sents, n):
    """context (n-1 words) -> Counter of the words that follow it."""
    foll = defaultdict(Counter)
    for s in sents:
        toks = ["<s>"] * (n - 1) + to_tokens(s) + ["</s>"]
        for i in range(len(toks) - n + 1):
            foll[tuple(toks[i:i + n - 1])][toks[i + n - 1]] += 1
    return foll

In [4]:
sents = to_sentences(open("corpus.txt").read())

## Sample, don't argmax

Always picking the single most likely word gets stuck repeating itself, so `generate` *samples*:
`rng.choices` picks each follower with chance proportional to its count. The context is the last `n-1` tokens.
A fixed seed makes every run reproducible. It stops at `</s>`, at an unseen context, or after `max_len` words.

In [5]:
def generate(foll, n, seed=0, max_len=25):
    rng = random.Random(seed)
    ctx = ("<s>",) * (n - 1); out = []
    for _ in range(max_len):
        c = foll[ctx]
        if not c:
            break
        words, weights = zip(*c.items())
        nxt = rng.choices(words, weights=weights)[0]
        if nxt == "</s>":
            break
        out.append(nxt)
        ctx = (ctx + (nxt,))[-(n - 1):]
    return " ".join(out)

## Bigram vs trigram, same four seeds

In [6]:
bi, tri = build(sents, 2), build(sents, 3)
print("BIGRAM (1 word of memory):")
for s in range(4):
    print("  -", generate(bi, 2, seed=s))
print("\nTRIGRAM (2 words of memory):")
for s in range(4):
    print("  -", generate(tri, 3, seed=s))

BIGRAM (1 word of memory):
  - he enjoys a good book make strong
  - the piano softly on the park after a good book start my day
  - rainy mornings make me want more coffee and coffee
  - the quiet and coffee

TRIGRAM (2 words of memory):
  - he enjoys a long walk on sunny weekends
  - the mountains are beautiful in the green park
  - rainy mornings make me want more coffee
  - the garden is full of flowers in the park is quiet and full of old books


The bigram text drifts ("make strong", "coffee and coffee") because each word only sees the one before it.
The trigram text reads like real sentences, but mostly because it is copying corpus phrases. With 60 sentences
almost every trigram was seen only once. More context means more fluency, but it needs much more data.

## Test yourself

**1. Why sample instead of always taking the most likely next word?**
Greedy choice is deterministic and loops easily, e.g. "the morning … the morning". Sampling in proportion to the
counts keeps likely words common but still gives variety.

**2. How does the context update after each word?**
`ctx = (ctx + (nxt,))[-(n - 1):]`: append the new word and keep only the last n-1 tokens. That's the Markov
window sliding forward.

**3. Why does the trigram output look so much like the corpus?**
Most two-word contexts were seen once with a single follower, so there's only one choice. The model replays
training sentences, switching only where two sentences share a word pair (e.g. "in the").

## My notes

The key idea: generation is just "sample, append, slide the window", the same loop as GPT. What surprised me is
how the trigram's nicer text is mostly memorisation. "the garden is full of flowers in the park is quiet…" is two
corpus sentences glued together where they share "in the".